In [1]:
import jax 
import jax.numpy as jnp
import jax.random as jrandom

In [2]:
from probjax.core.custom_primitives.random_variable import rv_p

In [3]:
sampling_to_log_prob = {
    '_normal': jax.scipy.stats.norm.logpdf,
    '_uniform': lambda x,a,b: jax.scipy.stats.uniform.logpdf(x, a, (b-a)),
    '_bernoulli': lambda x,p: jax.scipy.stats.bernoulli.logpmf(x, p),
    '_gamma': jax.scipy.stats.gamma.logpdf,
}

In [11]:
from sbi.inference import SNLE
import torch

In [13]:
inference = SNLE()
inference.append_simulations(torch.randn(100, 2), torch.randn(100, 2))
inference.train()

In [4]:
def g(key):
    x1 = jax.random.normal(key)
   # x2 = jax.random.uniform(key)
    x3 = jax.random.bernoulli(key)
    #x4 = jax.random.gamma(key, 2.)
    y = x1 + x3 #+ x2 #+ x3 + x4
    return y

In [5]:
jaxpr = jax.make_jaxpr(g)(jrandom.PRNGKey(0))

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [6]:
from jax.core import JaxprEqn

def update_eqn(eqn, name):
    sampling_fn_jaxpr = eqn.params["jaxpr"]
    sampling_name = eqn.params["name"]
    
    print(sampling_fn_jaxpr.jaxpr.constvars)
    try:
        log_prob_fn = sampling_to_log_prob[sampling_name]
    except KeyError:
        raise NotImplementedError(f"Sampling function {sampling_name} no log_prob implemented")
    invals = jax._src.core.safe_map(lambda x: x.aval, eqn.outvars)
    additional_invals = jax._src.core.safe_map(lambda x: x.aval, sampling_fn_jaxpr.jaxpr.invars[1:])
    log_prob_fn_jaxpr = jax.make_jaxpr(log_prob_fn)(*invals, *additional_invals)
    
    params = {"sampling_fn_jaxpr": sampling_fn_jaxpr, "log_prob_fn_jaxpr": log_prob_fn_jaxpr, "name": name}
    new_eqn = JaxprEqn(eqn.invars, eqn.outvars, rv_p, params, eqn.effects, eqn.source_info)
    return new_eqn

def trace_rv(jaxpr):
    for i in range(len(jaxpr.eqns)):
        eqn = jaxpr.eqns[i]
        if eqn.primitive is not rv_p and "name" in eqn.params:
            new_eqn = update_eqn(eqn, "x" + str(i))
            jaxpr.eqns[i] = new_eqn
    return jaxpr

In [7]:
new_jaxpr = trace_rv(jaxpr)

[]
[]


In [8]:
new_jaxpr

{ lambda ; a:u32[2]. let
    b:key<fry>[] = random_wrap[impl=fry] a
    c:f32[] = random_variable[
      log_prob_fn_jaxpr={ lambda ; d:f32[]. let
          e:f32[] = integer_pow[y=2] 1.0
          f:f32[] = mul 6.2831854820251465 e
          g:f32[] = log f
          h:f32[] = sub d 0.0
          i:f32[] = integer_pow[y=2] h
          j:f32[] = div i e
          k:f32[] = add g j
          l:f32[] = div k -2.0
        in (l,) }
      name=x1
      sampling_fn_jaxpr={ lambda ; m:key<fry>[]. let
          n:f32[] = pjit[
            jaxpr={ lambda ; o:key<fry>[]. let
                p:f32[] = pjit[
                  jaxpr={ lambda ; q:key<fry>[] r:f32[] s:f32[]. let
                      t:u32[] = random_bits[bit_width=32 shape=()] q
                      u:u32[] = shift_right_logical t 9
                      v:u32[] = or u 1065353216
                      w:f32[] = bitcast_convert_type[new_dtype=float32] v
                      x:f32[] = sub w 1.0
                      y:f32[] = sub s

In [8]:
f_traced = lambda k: jax.core.jaxpr_as_fun(new_jaxpr)(k)[0]

In [9]:
f_traced(jrandom.PRNGKey(0))

Array(0.794, dtype=float32)

In [10]:
from probjax.core import joint_sample, log_potential_fn

In [11]:
sampler = joint_sample(f_traced)

sampler(jrandom.PRNGKey(0))

{'x1': Array(-0.206, dtype=float32), 'x3': Array(True, dtype=bool)}

In [12]:
log_potential_fn(f_traced, jax.random.PRNGKey(0))(sampler(jrandom.PRNGKey(0)))

TypeError: Argument 'None' of type '<class 'NoneType'>' is not a valid JAX type